# Training Ground Truth: EDA and Iterative Cleaning

This notebook assesses and cleans only `dataset/train/train_ground_truth.tsv`.

## Findings from raw data

The file contains 2,206,821 labels, one per Source 1 record. There are 123,247 singleton rows with an empty match list. The remaining records have 1–11 matches. The file contains 3,693,619 Source 2 target IDs and 3,944,746 Source 3 target IDs. No duplicate target IDs were found within any row, and the labels cover all Source 1 records.

## Cleaning decision

Preserve every label and every match. Trim whitespace around IDs and canonicalize list separators; preserve empty lists as empty. The raw file remains unchanged. The three workflow stages profile the raw labels, write a compressed cleaned TSV, and verify row/schema preservation and formatting. Run from top to bottom. Output: `dataset/cleaned/train/train_ground_truth.cleaned.tsv.gz`.

## EDA and cleaning results from the run

Match-count distribution: 123,247 Source 1 entities have 0 matches, 119,157 have 1, 375,212 have 2, 530,841 have 3, 484,115 have 4, 321,957 have 5, 164,868 have 6, 63,968 have 7, 18,680 have 8, 4,205 have 9, 534 have 10, and 37 have 11. Target IDs comprise 3,693,619 Source 2 IDs and 3,944,746 Source 3 IDs.

Cleaning QA passed: all 2,206,821 rows and both columns were preserved; 123,247 empty match lists remain; no duplicate target IDs or ID-list whitespace issues were found.


In [ ]:
import csv
import gzip
from collections import Counter
from pathlib import Path

filename = 'train_ground_truth.tsv'
search_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in search_roots if (p / "dataset" / "train" / filename).is_file()), None)
if project_root is None:
    raise FileNotFoundError(f"Could not locate dataset/train/{filename} from {Path.cwd()}")
raw_path = project_root / "dataset" / "train" / filename
clean_dir = project_root / "dataset" / "cleaned" / "train"
clean_dir.mkdir(parents=True, exist_ok=True)
clean_path = clean_dir / (raw_path.stem + ".cleaned.tsv.gz")
print("Raw file:", raw_path)
print("Clean output:", clean_path)


In [ ]:
# Iteration 1: inspect label coverage pattern and the size of each match list.
row_count = 0
matches_per_source1 = Counter()
target_prefix_counts = Counter()
source1_rows_with_matches = Counter()
duplicate_target_rows = 0
samples = []
with raw_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    columns = reader.fieldnames
    for row in reader:
        row_count += 1
        target_ids = [value.strip() for value in (row.get("matched_entity_ids") or "").split(",") if value.strip()]
        matches_per_source1[len(target_ids)] += 1
        prefixes = {value[:2] for value in target_ids}
        for value in target_ids: target_prefix_counts[value[:2]] += 1
        for prefix in prefixes: source1_rows_with_matches[prefix] += 1
        duplicate_target_rows += len(target_ids) != len(set(target_ids))
        if len(samples) < 5: samples.append(row)
print("Columns:", columns)
print("Rows:", f"{row_count:,}")
print("Source 1 rows by match count:", dict(sorted(matches_per_source1.items())))
print("Target IDs by source prefix:", dict(target_prefix_counts))
print("Rows matched in each source:", dict(source1_rows_with_matches))
print("Rows with duplicate IDs within match list:", duplicate_target_rows)
print("Sample labels:")
for sample in samples: print(sample)


In [ ]:
# Iteration 2: trim IDs and rejoin list values with a single comma. Empty labels stay empty.
# This does not infer or remove matches; it only standardizes whitespace in the TSV fields.
clean_rows = 0
clean_duplicate_rows = 0
with raw_path.open("r", encoding="utf-8", newline="") as src, gzip.open(clean_path, "wt", encoding="utf-8", newline="") as dst:
    reader = csv.DictReader(src, delimiter="\t")
    writer = csv.DictWriter(dst, fieldnames=reader.fieldnames, delimiter="\t", lineterminator="\n")
    writer.writeheader()
    for row in reader:
        source1_id = (row.get("source1_entity_id") or "").strip()
        target_ids = [value.strip() for value in (row.get("matched_entity_ids") or "").split(",") if value.strip()]
        clean_duplicate_rows += len(target_ids) != len(set(target_ids))
        writer.writerow({"source1_entity_id": source1_id, "matched_entity_ids": ",".join(target_ids)})
        clean_rows += 1
print("Cleaned rows written:", f"{clean_rows:,}")
print("Rows with duplicate target IDs (preserved for audit):", clean_duplicate_rows)


In [ ]:
# Iteration 3: verify row/schema preservation, empty-label count, and ID-list formatting.
qa_rows = 0
qa_empty_lists = 0
qa_duplicate_rows = 0
qa_whitespace_issues = 0
with gzip.open(clean_path, "rt", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    output_columns = reader.fieldnames
    for row in reader:
        qa_rows += 1
        s1 = row.get("source1_entity_id") or ""
        field = row.get("matched_entity_ids") or ""
        qa_empty_lists += not bool(field)
        ids = [value for value in field.split(",") if value]
        qa_duplicate_rows += len(ids) != len(set(ids))
        qa_whitespace_issues += s1 != s1.strip() or any(value != value.strip() for value in ids)
assert qa_rows == row_count, f"Row count changed: raw={{row_count}}, clean={{qa_rows}}"
assert output_columns == columns
assert qa_whitespace_issues == 0
print("Post-clean QA: PASS")
print("Rows preserved:", f"{qa_rows:,}")
print("Empty match lists (singletons):", f"{qa_empty_lists:,}")
print("Duplicate target rows:", qa_duplicate_rows)
